In [1]:
%load_ext autoreload
%autoreload 2
import sys
import pandas as pd
import os
import matplotlib.pyplot as plt
import cortex
import seaborn as sns
from os.path import join
from collections import defaultdict
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import dvu
from neuro.flatmaps_helper import load_flatmaps
from neuro.features.questions.gpt4 import QS_35_STABLE
import sys
from sklearn.metrics import pairwise_distances
import warnings
sys.path.append('../notebooks')
from tqdm import tqdm
from neuro import config
from neuro import analyze_helper
import neuro.viz
from neuro.features.qa_questions import get_questions, get_merged_questions_v3_boostexamples
# flatmaps_per_question = __import__('06_flatmaps_per_question')
# import viz
# import gct
from neuro.flatmaps_helper import load_flatmaps
from statsmodels.stats.multitest import multipletests

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/spacy/cli/_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/weasel/util/config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string


Note, this notebook requires first running `03_export_qa_flatmaps.ipynb` into `df_qa_dict.pkl` files for each subject.

### load gemv average flatmaps

In [4]:
subject = 'S02'
# subject = 'S03'
gemv_flatmaps_dict_S02, gemv_flatmaps_dict_S03 = load_flatmaps(
    normalize_flatmaps=False, load_timecourse=False)
if subject == 'S02':
    gemv_flatmaps_dict = gemv_flatmaps_dict_S02
elif subject == 'S03':
    gemv_flatmaps_dict = gemv_flatmaps_dict_S03
df_gct = pd.DataFrame(gemv_flatmaps_dict).T
df_gct = df_gct[~np.array([x[0] in ['START', 'END'] for x in df_gct.index])]

In [5]:
def get_eng1000_weight_for_subject(subject):
    data = joblib.load(join(config.RESULTS_DIR_LOCAL, 'results_best_ensemble.pkl'))
    rr, cols_varied, mets = data['r'], data['cols_varied'], data['mets']
    metric_sort = 'corrs_tune_pc_weighted_mean'
    
    r = rr[
        (rr.feature_space == 'eng1000') * \
        (rr.num_stories == -1) * \
        (rr.feature_selection_alpha == -1) * \
        (rr.ndelays == 4)
    ]

    args = r[r.subject == subject].iloc[0]
    model_params = joblib.load(
        join(args.save_dir_unique, 'model_params.pkl'))
    print(args.feature_space, args.pc_components, args.ndelays)
    wt = model_params['weights'] # wt is (n_delays x n_features) x n_voxels
    n_features = wt.shape[0] / args.ndelays
    wt = wt.reshape(args.ndelays, int(n_features), -1)
    wt = wt.mean(axis=0) # average over delays

    return wt, args['corrs_test']

wt, corrs_test = get_eng1000_weight_for_subject(subject)
print('enc perf', np.mean(corrs_test))
# get top-10 percentile mask of corrs_test
mask_top = corrs_test >= np.percentile(corrs_test, 90)

/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.4.2 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


eng1000 100 4
enc perf 0.09502538966591687


In [253]:
ENG1000_WORDS = neuro.analyze_helper.ENG1000_WORDS
# got these mappings by prompting GPT to """Return a dictionary where that maps each query to the closest semantic match in the database. Make sure every key is exactly in the queries list and every value is exactly in the database. Think hard."""
mappings_S02 = {
  'birthdays': 'birthday',
  'communication': 'talk',
  'death': 'die',
  'emotion': 'feel',
  'emotional expression': 'feel',
  'food preparation': 'cook',
  'hair and clothing': 'clothes',
  'laughter': 'laugh',
  'locations': 'place',
  'measurements': 'measure',
  'moments': 'moment',
  'negativity': 'bad',
  'physical injury or trauma': 'hurt',
  'rejection': 'no',
  'surprise': 'surprise',
  'time': 'hour',
  'abstract descriptions': 'idea',
  'cultural references': 'art',
  'dialogue': 'talk',
  'industry or profession': 'business',
  'negations': 'not',
  'numbers': 'number',
  'opinions or judgments': 'think',
  'personal or interactions interactions': 'person',
  'personal reflections or thoughts': 'think',
  'personal values or beliefs': 'trust',
  'physical actions': 'act',
  'planning or organizing': 'order',
  'proper nouns': 'name',
  'relationships between people': 'family',
  'sensory experiences': 'taste',
  'specific objects or items': 'object',
  'technical or specialized terminology': 'computer',
  'Body parts': 'body',
  'Descriptive elements of scenes or objects': 'picture',
  'Direction and location descriptions': 'direction',
  'Location names': 'city',
  'Personal growth and reflection': 'develop',
  'Scenes and settings': 'place',
  'Spatial positioning and directions': 'position',
  'Time and numbers': 'number',
  'Travel and location names': 'travel',
  'Unappetizing foods': 'poison',
  'Verbal interactions': 'speak',
  'Clothing and Physical Appearance': 'clothes',
  'Colors': 'colour',
  'Dialogue': 'talk',
  'Fear and Avoidance': 'fear',
  'Gruesome body imagery': 'blood',
  'Introspection': 'mind',
  'Measurements': 'measure',
  'Negative Emotional Reactions': 'sad',
  'Numbers': 'number',
  'Positive Emotional Reactions': 'happy',
  'Professions and Personal Backgrounds': 'job',
  'Recognition': 'notice',
  'Relationships': 'family',
  'Secretive Or Covert Actions': 'hide',
  'Sexual and Romantic Interactions': 'sex',
  'Times': 'hour',
  'Years': 'year'
}

mappings_S03 = {
  "action or movement": "move",
  "age": "age",
  "agreement and questioning": "agree",
  "body language": "body",
  "communication": "speak",
  "conflict resolution": "peace",
  "family and relationships": "family",
  "food and drinks": "drink",
  "locations": "place",
  "love and joy": "love",
  "movement or action": "activity",
  "negative experiences": "trouble",
  "numbers": "number",
  "numbers or measurements": "measure",
  "physical injury": "hurt",
  "vomiting, sickness": "sick",
  "Clothing and Physical Appearance": "clothes",
  "Colors": "colour",
  "Dialogue": "talk",
  "Direction and location descriptions": "direction",
  "Gruesome body imagery": "blood",
  "Introspection": "think",
  "Measurements": "measure",
  "Numbers": "number",
  "Relationships": "partner",
  "Scenes and settings": "place",
  "Times": "hour",
  "Travel and location names": "travel",
  "Years": "year",
  "Body parts": "body",
  "Conversational transitions": "then",
  "Descriptive elements of scenes or objects": "shape",
  "Dialogue and responses": "reply",
  "Fear and Avoidance": "afraid",
  "Garbage, food, and household items": "waste",
  "Location names": "city",
  "Negative Emotional Reactions": "sad",
  "Positive Emotional Reactions": "happy",
  "Professions and Personal Backgrounds": "job",
  "Recognition": "remember",
  "Secretive Or Covert Actions": "hide",
  "Self-reflection and growth": "learn",
  "Sexual and Romantic Interactions": "sex",
}


if subject == 'S02':
    mappings = mappings_S02
elif subject == 'S03':
    mappings = mappings_S03

ks = [k[0] for k in df_gct.index.tolist()]
for k, v in mappings.items():
    assert k in ks
    assert v in ENG1000_WORDS, v

for k in ks:
    assert k in mappings.keys(), k

In [254]:
corrs = defaultdict(list)
for i in range(len(df_gct)):
    row = df_gct.iloc[i]
    word = df_gct.index[i][0]
    eng1000_word = mappings[word]
    eng1000_idx = ENG1000_WORDS.index(eng1000_word)
    weight_vector = wt[eng1000_idx, :]  # shape: n_voxels
    gct_vector = row.values  # shape: n_voxels

    # apply mask
    weight_vector = weight_vector[mask_top]
    gct_vector = gct_vector[mask_top]

    corr = np.corrcoef(gct_vector, weight_vector)[0,1]
    corrs['corr'].append(corr)
    corrs['word'].append(word)
    corrs['eng1000_word'].append(eng1000_word)
    corrs['expt'].append(df_gct.index[i][1])
    # print(f'Word: {word}, ENG1000 word: {eng1000_word}, Corr: {corr:.4f}')
corrs = pd.DataFrame(corrs)

# filter out non-GCT expts
corrs = corrs[(corrs.expt.notna()) & ~(corrs.expt == 'qa')].sort_values(by='corr', ascending=False)
corrs

,corr,word,eng1000_word,expt
3,0.706835,agreement and questioning,agree,458.0
9,0.526790,locations,place,342.0
11,0.221919,movement or action,activity,152.0
15,0.215148,physical injury,hurt,148.0
12,0.166807,negative experiences,trouble,403.0
8,0.136135,food and drinks,drink,466.0
4,0.132393,body language,body,99.0
0,0.097141,action or movement,move,158.0
2,0.045445,age,age,160.0
14,-0.088280,numbers or measurements,measure,408.0


In [255]:
# remove bad matches
if subject == 'S03':
    corrs = corrs[~corrs.word.isin([
        'conflict resolution', 'communication', 'movement or action', 'vomiting, sickness', 
    ])]
elif subject == 'S02':
    corrs = corrs[~corrs.word.isin([
        'rejection', 'emotion', 'emotional expression', 'time', 'communication'
    ])]
# visualize whole df
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(corrs.sort_values(by='corr', ascending=False))

,corr,word,eng1000_word,expt
3,0.706835,agreement and questioning,agree,458.0
9,0.526790,locations,place,342.0
15,0.215148,physical injury,hurt,148.0
12,0.166807,negative experiences,trouble,403.0
8,0.136135,food and drinks,drink,466.0
4,0.132393,body language,body,99.0
0,0.097141,action or movement,move,158.0
2,0.045445,age,age,160.0
14,-0.088280,numbers or measurements,measure,408.0
10,-0.093003,love and joy,love,337.0


In [257]:
# compute all pairwise correlations
dists = pairwise_distances(
    wt[:, mask_top],
    # wt[:, :],
    metric='correlation',
    n_jobs=-1,
)
# take upper diag of dists
dists = dists[np.triu_indices(dists.shape[0], k=1)]
dists = 1 - dists  # convert to correlations
# plt.hist(dists)

n_samples = len(corrs['corr'])
avg_corr = np.mean(corrs['corr'])

# compute p-value from null distribution
means = []
np.random.seed(42)
for i in tqdm(range(1000)):
    sampled_idxs = np.random.choice(len(dists), n_samples, replace=True)
    means.append(np.mean(dists[sampled_idxs]))
p_value = np.mean(np.array(means) >= avg_corr)
print(f'Average correlation: {avg_corr:.4f}, p-value: {p_value:.4f}')

100%|██████████| 1000/1000 [00:00<00:00, 20542.89it/s]

Average correlation: 0.0945, p-value: 0.2130


In [258]:
corrs_print = corrs[['word', 'eng1000_word', 'corr']].rename(
    columns={'word': 'GCT explanation', 'eng1000_word': 'Eng1000 word', 'corr': 'Correlation'}
).round(3).sort_values(by='Correlation', ascending=False)
print(corrs_print.style.format(precision=3).hide(axis="index").to_latex(hrules=True))

\begin{tabular}{llr}
\toprule
GCT explanation & Eng1000 word & Correlation \\
\midrule
agreement and questioning & agree & 0.707 \\
locations & place & 0.527 \\
physical injury & hurt & 0.215 \\
negative experiences & trouble & 0.167 \\
food and drinks & drink & 0.136 \\
body language & body & 0.132 \\
action or movement & move & 0.097 \\
age & age & 0.045 \\
numbers or measurements & measure & -0.088 \\
love and joy & love & -0.093 \\
age & age & -0.114 \\
family and relationships & family & -0.118 \\
numbers & number & -0.385 \\
\bottomrule
\end{tabular}

